# DFT Performance Breakdown

This notebook runs the DFT SCF benchmark on compact grids and plots where time is going. The goal is not peak throughput yet; it is to build evidence for the first serious MLX/Metal optimization target.

In [ ]:
import matplotlib.pyplot as plt

from mlx_atomistic.benchmarks.dft_scf import build_payload

The benchmark emits one row per grid and mixer. Each row includes runtime metadata, final residual, total time, and section timings from the SCF loop.

In [ ]:
payload = build_payload(grid_shape=(4, 4, 4), sizes="4,8", iterations=2, mixer="both")
rows = payload["cases"]
[{k: row[k] for k in ["grid_shape", "mixer", "ms_per_iteration", "final_residual"]} for row in rows]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

for mixer in sorted({row["mixer"] for row in rows}):
    subset = [row for row in rows if row["mixer"] == mixer]
    axes[0].plot(
        [row["grid_points"] for row in subset],
        [row["ms_per_iteration"] for row in subset],
        marker="o",
        label=mixer,
    )

axes[0].set_xlabel("grid points")
axes[0].set_ylabel("ms / SCF iteration")
axes[0].set_title("grid scaling")
axes[0].legend()

row = rows[-1]
timing_keys = ["hartree_ms", "xc_ms", "solver_ms", "kinetic_ms", "mixer_ms", "force_ms"]
axes[1].bar(timing_keys, [row["timings"][key] for key in timing_keys])
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_ylabel("cumulative ms")
axes[1].set_title(f"timing breakdown {row['grid_shape']} / {row['mixer']}");